## Bureau Feature Engineering

In [11]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [12]:
import pandas as pd
import numpy as np
from src.data_utils import load_raw, save_interim

In [34]:
df_bureau = load_raw("bureau.csv")

### Base Aggregation (Loan Count)

In [35]:
bureau_loan_count = df_bureau.groupby('SK_ID_CURR').agg({
    'SK_ID_BUREAU':'count'
})

### Credit Status Counts + Ratios

In [36]:
credit_status=df_bureau.groupby(
    ['SK_ID_CURR', 'CREDIT_ACTIVE'])['SK_ID_BUREAU'].count().unstack(fill_value=0)

credit_status.columns = [
    'BUREAU_' + col.upper().replace(' ','_') + '_COUNT' 
    for col in credit_status.columns]


total_loan = credit_status.sum(axis=1)

credit_status['BUREAU_ACTIVE_RATIO'] = (
    credit_status.get('BUREAU_ACTIVE_COUNT', 0)/
    total_loan.replace(0,np.nan)
)

credit_status['BUREAU_BAD_DEBT_RATIO'] = (
    credit_status.get('BUREAU_BAD_DEBT_COUNT', 0)/
    total_loan.replace(0,np.nan)
)

### Credit Exposure & Utilization

In [37]:
credit_exposure = df_bureau.groupby('SK_ID_CURR').agg(
    BUREAU_TOTAL_CREDIT = ('AMT_CREDIT_SUM', 'sum',),
    BUREAU_TOTAL_DEBT = ('AMT_CREDIT_SUM_DEBT', 'sum')
)

credit_exposure['BUREAU_UTILIZATION_RATIO'] = (
    credit_exposure['BUREAU_TOTAL_DEBT']/
    credit_exposure['BUREAU_TOTAL_CREDIT'].replace(0, np.nan)
)

### Overdue Features

In [38]:
overdue_agg = df_bureau.groupby('SK_ID_CURR').agg(
    BUREAU_MAX_DAYS_OVERDUE = ('CREDIT_DAY_OVERDUE', 'max'),
    BUREAU_MEAN_DAYS_OVERDUE = ('CREDIT_DAY_OVERDUE', 'mean'),
    BUREAU_SUM_DAYS_OVERDUE = ('CREDIT_DAY_OVERDUE', 'sum'),
    BUREAU_MAX_OVERDUE_AMOUNT = ('AMT_CREDIT_SUM_OVERDUE', 'max'),
    BUREAU_MEAN_OVERDUE_AMOUNT = ('AMT_CREDIT_SUM_OVERDUE', 'mean'),
    BUREAU_TOTAL_OVERDUE_SUM = ('AMT_CREDIT_SUM_OVERDUE', 'sum')
)

### Overdue Behavior Indicator

In [39]:
df_bureau['HAS_OVERDUE'] = (df_bureau['CREDIT_DAY_OVERDUE']>0).astype(int)

overdue_flag = df_bureau.groupby('SK_ID_CURR').agg(
    BUREAU_OVERDUE_RATIO = ('HAS_OVERDUE', 'mean')
)

### Recency Features

In [21]:
df_bureau['DAYS_CREDIT_POS'] = -df_bureau['DAYS_CREDIT']

credit_days = df_bureau.groupby('SK_ID_CURR').agg(
    BUREAU_MIN_DAYS_CREDIT = ('DAYS_CREDIT_POS', 'min'),
    BUREAU_MAX_DAYS_CREDIT = ('DAYS_CREDIT_POS', 'max'),
    BUREAU_MEAN_DAYS_CREDIT = ('DAYS_CREDIT_POS', 'mean')
)

### Merge and Save

In [40]:
bureau_final = (bureau_loan_count
    .merge(credit_status, on='SK_ID_CURR', how='left')
    .merge(credit_exposure, on='SK_ID_CURR', how='left')
    .merge(overdue_agg, on='SK_ID_CURR', how='left')
    .merge(overdue_flag, on='SK_ID_CURR', how='left')
    .merge(credit_days, on='SK_ID_CURR', how='left')
)
bureau_final.reset_index(inplace=True)

In [ ]:
save_interim(bureau_final, 'bureau_agg.csv')